# Parking-capacity — entraînement YOLOv8-seg satellite (APKLOT)

Ajustez **`ZIP_PATH`**, **`OUTPUT_RUN`** (sur Drive pour ne pas perdre les checkpoints), **`SAVE_PERIOD_EPOCHS`** (checkpoint tous les N epochs ; 0 = désactivé).

## A. Monter Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Chemin vers parking_capacity_colab.zip sur votre Drive
ZIP_PATH = "/content/drive/MyDrive/parking_capacity_colab.zip"
ROOT = "/content/parking_colab"
# Sortie entraînement sur Drive : checkpoints Ultralytics y sont écrits directement
OUTPUT_RUN = "/content/drive/MyDrive/colab_runs/yolo_seg"
SAVE_PERIOD_EPOCHS = 5  # 0 pour désactiver ; sinon checkpoint tous les N epochs dans OUTPUT_RUN/yolo_train/weights/
EPOCHS = 50
MODEL = "yolov8m-seg.pt"
IMGSZ = 640
BENCHMARK_MOSAIC = "/content/drive/MyDrive/colab_benchmark_mosaic.png"

## B. Décompresser `parking_capacity_colab.zip`

In [ ]:
import os
import shutil
import subprocess

shutil.rmtree(ROOT, ignore_errors=True)
os.makedirs(os.path.dirname(ROOT), exist_ok=True)
subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", ROOT], check=True)
os.chdir(os.path.join(ROOT, "project_snapshot"))
print("cwd:", os.getcwd())

## C. Installation propre du package
Désinstalle l’ancienne version pip puis réinstalle en mode éditable avec les extras vision / satellite.

In [ ]:
!pip uninstall -y parking-capacity
!pip install -q -r ../requirements_colab.txt
!pip install -q -e ".[train_satellite,vision]"

## D. Environnement Colab (Python, CUDA, torch, Ultralytics, GPU)

In [ ]:
import subprocess
import sys

print("Python:", sys.version)

try:
    import torch

    print("torch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA version (torch):", torch.version.cuda)
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch import:", e)

try:
    import ultralytics

    print("ultralytics:", ultralytics.__version__)
except Exception as e:
    print("ultralytics:", e)

subprocess.run(["nvidia-smi"], check=False)

## E. Validation post-install (synchronisation notebook ↔ package)
Vérifie la présence des options CLI attendues dans les textes d’aide.

In [ ]:
import subprocess


def _help(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return (r.stdout or "") + (r.stderr or "")


helps = {
    "parking-capacity --help": _help(["parking-capacity", "--help"]),
    "parking-capacity datasets-prepare --help": _help(
        ["parking-capacity", "datasets-prepare", "--help"]
    ),
    "parking-capacity inspect-dataset --help": _help(
        ["parking-capacity", "inspect-dataset", "--help"]
    ),
}

missing = []
if "benchmark-dataset-mosaics" not in helps["parking-capacity --help"]:
    missing.append(("parking-capacity --help", "benchmark-dataset-mosaics"))
if "--apklot-view" not in helps["parking-capacity datasets-prepare --help"]:
    missing.append(("datasets-prepare --help", "--apklot-view"))

train_h = _help(["parking-capacity", "train-yolo-seg", "--help"])
if "--force-incompatible-dataset" not in train_h:
    missing.append(("train-yolo-seg --help", "--force-incompatible-dataset"))

if missing:
    for ctx, token in missing:
        print(f"Manquant dans {ctx}: {token}")
    raise RuntimeError(
        "Le notebook et le package exporté ne sont pas synchronisés. "
        "Refaites un export local : parking-capacity export-colab-training … puis ré-uploadez le ZIP."
    )
print("Validation CLI : OK")

## F. Statistiques du jeu préparé

In [ ]:
!parking-capacity dataset-stats --dataset apklot

## G. Téléchargement / préparation APKLOT (vue satellite)
Si le ZIP ne contenait pas les données (`DATASET_TOO_LARGE.txt`) ou si les dossiers manquent, exécutez ces lignes (peut être long).

In [ ]:
# !parking-capacity datasets-download --dataset apklot
!parking-capacity datasets-prepare --dataset apklot --apklot-view satellite

## H. Inspection APKLOT (`dataset_type`, satellite, garde-fous)
Après préparation, `dataset_prepare_meta.json` indique si l’entraînement « vue satellite » est pertinent.

In [ ]:
import json
import subprocess

r = subprocess.run(
    ["parking-capacity", "inspect-dataset", "--dataset", "apklot"],
    check=True,
    capture_output=True,
    text=True,
)
info = json.loads(r.stdout)
print(json.dumps(info, indent=2, ensure_ascii=False))

dt = info.get("dataset_type")
pm = info.get("prepare_meta") or {}
sat_ok = pm.get("satellite_segmentation_suitable")
by_view = pm.get("images_by_view") or {}
n_sat = int(by_view.get("satellite", 0))

print("dataset_type:", dt)
print("satellite_segmentation_suitable:", sat_ok)
print("images_by_view:", by_view)
print("images satellite (préparé):", n_sat)

ALLOW_SAT_TRAIN = bool(sat_ok) if sat_ok is not None else True
if sat_ok is False:
    print(
        "\n>>> Diagnostic : APKLOT préparé sans image satellite exploitable pour la segmentation orthophoto. "
        "Vérifiez raw/apklot (« 1. Satellite », git lfs pull) ou utilisez --apklot-view all en connaissance de cause, "
        "ou un jeu DOTA/xView/SpaceNet.\n"
    )

## I. Mosaïque benchmark (jeux satellite)

In [ ]:
import os
import subprocess

os.makedirs(os.path.dirname(BENCHMARK_MOSAIC), exist_ok=True)
subprocess.run(
    [
        "parking-capacity",
        "benchmark-dataset-mosaics",
        "--out",
        BENCHMARK_MOSAIC,
        "--datasets",
        "apklot,dota,xview,spacenet",
        "--samples",
        "4",
    ],
    check=True,
)
print("Mosaïque écrite :", BENCHMARK_MOSAIC)

## J. Entraînement YOLOv8-seg
Bloqué si `satellite_segmentation_suitable` est faux (section H). Pour ignorer : ajoutez `--force-incompatible-dataset` dans la commande ci-dessous.

In [ ]:
import os
import subprocess

if not ALLOW_SAT_TRAIN:
    raise RuntimeError(
        "Entraînement satellite bloqué : satellite_segmentation_suitable=false. "
        "Corrigez les données APKLOT ou utilisez explicitement --force-incompatible-dataset si vous assumez le risque."
    )

cmd = [
    "parking-capacity",
    "train-yolo-seg",
    "--dataset",
    "apklot",
    "--model",
    MODEL,
    "--epochs",
    str(EPOCHS),
    "--imgsz",
    str(IMGSZ),
    "--output-dir",
    OUTPUT_RUN,
    "--save-period",
    str(SAVE_PERIOD_EPOCHS),
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("Checkpoints :", os.path.join(OUTPUT_RUN, "yolo_train", "weights"))

## K. Reprise après interruption (`--resume`)

In [ ]:
import os
import subprocess

LAST_PT = os.path.join(OUTPUT_RUN, "yolo_train", "weights", "last.pt")
# Décommenter pour reprendre :
# subprocess.run(
#     [
#         "parking-capacity",
#         "train-yolo-seg",
#         "--dataset",
#         "apklot",
#         "--resume",
#         "--weights",
#         LAST_PT,
#         "--epochs",
#         str(EPOCHS),
#         "--imgsz",
#         str(IMGSZ),
#         "--output-dir",
#         OUTPUT_RUN,
#         "--save-period",
#         str(SAVE_PERIOD_EPOCHS),
#     ],
#     check=True,
# )
print("last.pt attendu:", LAST_PT)

## L. Évaluation val / test (Ultralytics)

In [ ]:
import glob

weights_glob = OUTPUT_RUN + "/**/weights/best.pt"
best_list = sorted(glob.glob(weights_glob, recursive=True))
assert best_list, "Aucun best.pt trouvé sous OUTPUT_RUN"
BEST_PT = best_list[-1]

yaml_candidates = [
    "../datasets/prepared/apklot/yolo_seg_dataset/dataset.yaml",
    "data/datasets/prepared/apklot/yolo_seg_dataset/dataset.yaml",
]
import os as _os

DATA_YAML = next((p for p in yaml_candidates if _os.path.isfile(p)), yaml_candidates[0])
print("BEST_PT", BEST_PT)
print("DATA_YAML", DATA_YAML)

In [ ]:
import subprocess

subprocess.run(
    ["yolo", "segment", "val", f"model={BEST_PT}", f"data={DATA_YAML}", "split=val"],
    check=True,
)
subprocess.run(
    ["yolo", "segment", "val", f"model={BEST_PT}", f"data={DATA_YAML}", "split=test"],
    check=True,
)

## M. Exporter résultats vers Drive (copie explicite)

In [ ]:
import os
import shutil
from pathlib import Path

EXPORT_DIR = "/content/drive/MyDrive/colab_yolo_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

bp = Path(BEST_PT).resolve()
parts = bp.parts
if "yolo_train" in parts:
    i = parts.index("yolo_train")
    run_root = Path(*parts[:i])
else:
    run_root = bp.parent.parent

yt = run_root / "yolo_train"
scan_dirs = [yt, run_root]
for base in scan_dirs:
    for rel in ("results.csv", "args.yaml", "train_metrics.json"):
        src = base / rel
        if src.is_file():
            shutil.copy2(src, os.path.join(EXPORT_DIR, rel))

weights_dir = bp.parent
for w in ("best.pt", "last.pt"):
    p = weights_dir / w
    if p.is_file():
        shutil.copy2(p, os.path.join(EXPORT_DIR, w))

tm = run_root / "train_metrics.json"
if tm.is_file():
    shutil.copy2(tm, os.path.join(EXPORT_DIR, "train_metrics_run.json"))

for name in ("sample_overlays", "sample_masks"):
    sd = run_root / name
    if sd.is_dir():
        shutil.copytree(sd, os.path.join(EXPORT_DIR, name), dirs_exist_ok=True)

print("Export →", EXPORT_DIR)

## N. Test orthophoto réelle

In [ ]:
import os
import subprocess

TEST_OUT = "/content/drive/MyDrive/colab_test_seg"
os.makedirs(TEST_OUT, exist_ok=True)
subprocess.run(
    [
        "parking-capacity",
        "test-segmentation-real",
        "--address",
        "2 Bd Industriel, 76270 Neufchâtel-en-Bray",
        "--weights",
        BEST_PT,
        "--out",
        TEST_OUT,
    ],
    check=True,
)

## O. Cohérence build (optionnel)
Compare la version pip au fichier `build_info.json` extrait dans le ZIP.

In [ ]:
!parking-capacity doctor-build --export-dir /content/parking_colab

## P (optionnel). Détection véhicules — vue aérienne / satellite (Kaggle ou Hugging Face)

Télécharge un jeu **détection** (bbox YOLO), pas la segmentation parking APKLOT. Utile pour un modèle « voitures vues du ciel ».

**Kaggle** : secrets Colab `KAGGLE_USERNAME` / `KAGGLE_KEY`, ou `~/.kaggle/kaggle.json`, ou les variables **visibles** dans la cellule suivante (`KAGGLE_*_VISIBLE`) pour une session de test — **ne sauvegardez pas** le notebook avec de vrais secrets si vous le versionnez sur Git.

**Hugging Face** : secret `HF_TOKEN`, ou variable **`HF_TOKEN_VISIBLE`** dans la cellule suivante ; les jeux publics passent souvent sans token.

L’entraînement utilise **`yolo detect train`** (Ultralytics). Le sous-commande `parking-capacity train-vehicle-detector` du paquet reste liée au registre **xView** local ; ici on pointe vers un `dataset.yaml` / `data.yaml` téléchargé.

In [ ]:
# --- Réglages téléchargement + entraînement détection véhicules ---
import os

# Jetons / user visibles (Colab test). Laisser "" pour utiliser Secrets Colab ou l’env déjà défini.
# Exemple test (fictif) : KAGGLE_USERNAME_VISIBLE = "demo_user"
KAGGLE_USERNAME_VISIBLE = ""
KAGGLE_KEY_VISIBLE = ""
HF_TOKEN_VISIBLE = ""

if KAGGLE_USERNAME_VISIBLE.strip() and KAGGLE_KEY_VISIBLE.strip():
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME_VISIBLE.strip()
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY_VISIBLE.strip()
if HF_TOKEN_VISIBLE.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN_VISIBLE.strip()

# "kaggle" | "huggingface"
AERIAL_VEHICLE_MODE = "kaggle"

# Kaggle (ex. jeu déjà formaté YOLO v5/v8)
KAGGLE_DATASET_SLUG = "braunge/aerial-view-car-detection-for-yolov5"

# Hugging Face (ex. orthomosaïque + annotations YOLO)
HF_VEHICLE_DATASET_ID = "titoruizh/Drone-Orthomosaic-Vehicles-Yolo-annotation"

AERIAL_DATA_ROOT = "/content/aerial_vehicle_dataset"
OUTPUT_VEHICLE_DET = "/content/drive/MyDrive/colab_runs/yolo_vehicle_aerial"
VEHICLE_EPOCHS = 30
VEHICLE_MODEL = "yolov8m.pt"
VEHICLE_IMGSZ = 640
VEHICLE_SAVE_PERIOD = 5  # 0 = défaut Ultralytics

!pip install -q kaggle huggingface_hub pyyaml

In [ ]:
import json
import os
import shutil
import subprocess
import zipfile
from pathlib import Path
from typing import Optional

import yaml

_IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def _unwrap_single_dataset_folder(root: Path) -> Path:
    """Kaggle extrait souvent un seul sous-dossier sans yaml à la racine : on descend dedans."""
    r = root.resolve()
    for _ in range(5):
        if not r.is_dir():
            return r
        if _find_yaml_under(r) is not None:
            return r
        subs = [p for p in r.iterdir() if p.is_dir() and p.name not in (".ipynb_checkpoints", "__MACOSX")]
        files = [p for p in r.iterdir() if p.is_file()]
        if len(subs) == 1 and len(files) <= 2 and all(f.suffix.lower() in (".zip", ".txt", "") for f in files):
            r = subs[0]
            continue
        if len(subs) == 1 and not any(r.glob("*.yaml")) and not any(r.glob("*.yml")):
            r = subs[0]
            continue
        break
    return r


def _yaml_looks_yolo_data(p: Path) -> bool:
    try:
        data = yaml.safe_load(p.read_text(encoding="utf-8"))
    except Exception:
        return False
    if not isinstance(data, dict):
        return False
    return ("train" in data) and ("names" in data or "nc" in data)


def _find_yaml_under(root: Path) -> Optional[Path]:
    for name in ("dataset.yaml", "data.yaml"):
        found = sorted(root.rglob(name), key=lambda x: len(x.parts))
        if found:
            return found[0]
    for p in sorted(root.rglob("*.yaml"), key=lambda x: len(x.parts)):
        if _yaml_looks_yolo_data(p):
            return p
    for p in sorted(root.rglob("*.yml"), key=lambda x: len(x.parts)):
        if _yaml_looks_yolo_data(p):
            return p
    return None


def _dir_has_images(d: Path) -> bool:
    if not d.is_dir():
        return False
    for p in d.iterdir():
        if p.is_file() and p.suffix.lower() in _IMG_EXT:
            return True
    return False


def _dir_has_yolo_labels(d: Path) -> bool:
    if not d.is_dir():
        return False
    return any(p.suffix.lower() == ".txt" for p in d.iterdir() if p.is_file())


def _collect_split_dirs(root: Path):
    """Retourne (train_img, train_lbl, val_img, val_lbl) ou None."""
    train_img = train_lbl = val_img = val_lbl = None
    for split in ("train", "Train", "TRAIN"):
        ti, tl = root / split / "images", root / split / "labels"
        if _dir_has_images(ti) and _dir_has_yolo_labels(tl):
            train_img, train_lbl = ti, tl
            break
    if train_img is None:
        return None
    for split in ("valid", "val", "test", "Valid", "Val", "Test"):
        vi, vl = root / split / "images", root / split / "labels"
        if _dir_has_images(vi) and _dir_has_yolo_labels(vl):
            val_img, val_lbl = vi, vl
            break
    if val_img is None:
        val_img, val_lbl = train_img, train_lbl
    return train_img, train_lbl, val_img, val_lbl


def _find_parallel_images_labels(root: Path):
    """Repère un couple images/labels avec le même parent (ex. split/images + split/labels)."""
    for labels_dir in root.rglob("labels"):
        if not labels_dir.is_dir() or not _dir_has_yolo_labels(labels_dir):
            continue
        par = labels_dir.parent
        images_dir = par / "images"
        if _dir_has_images(images_dir):
            return images_dir, labels_dir
    return None


def _ensure_minimal_yolo_yaml(root: Path) -> Path:
    """Construit un dataset.yaml si le dépôt Kaggle/HF n’en fournit pas un au bon endroit."""
    tr_i, tr_l = root / "images" / "train", root / "labels" / "train"
    va_i, va_l = root / "images" / "val", root / "labels" / "val"
    if _dir_has_images(tr_i) and _dir_has_yolo_labels(tr_l):
        val_use = (
            va_i
            if _dir_has_images(va_i) and _dir_has_yolo_labels(va_l)
            else tr_i
        )
        out = root / "_colab_vehicle_dataset.yaml"
        body = {
            "path": str(root.resolve()),
            "train": str(tr_i.relative_to(root)),
            "val": str(val_use.relative_to(root)),
            "nc": 1,
            "names": {0: "vehicle"},
        }
        out.write_text(yaml.safe_dump(body, sort_keys=False), encoding="utf-8")
        return out
    fixed_pairs = [
        (root / "images" / "train", root / "labels" / "train"),
        (root / "images" / "val", root / "labels" / "val"),
        (root / "train" / "images", root / "train" / "labels"),
        (root / "Data" / "images" / "train", root / "Data" / "labels" / "train"),
        (root / "data" / "images" / "train", root / "data" / "labels" / "train"),
        (root / "datasets" / "images" / "train", root / "datasets" / "labels" / "train"),
        (root / "Aerial-Cars-1" / "train" / "images", root / "Aerial-Cars-1" / "train" / "labels"),
    ]
    splits = _collect_split_dirs(root)
    if splits:
        train_img, train_lbl, val_img, val_lbl = splits
        out = root / "_colab_vehicle_dataset.yaml"
        body = {
            "path": str(root.resolve()),
            "train": str(train_img.relative_to(root)),
            "val": str(val_img.relative_to(root)),
            "nc": 1,
            "names": {0: "vehicle"},
        }
        out.write_text(yaml.safe_dump(body, sort_keys=False), encoding="utf-8")
        return out
    par = _find_parallel_images_labels(root)
    if par:
        img_d, lbl_d = par
        ds_root = root.resolve()
        try:
            tr_rel = str(img_d.relative_to(ds_root))
            va_rel = tr_rel
        except ValueError:
            ds_root = img_d.parent.parent.resolve() if (img_d.parent / "labels").is_dir() else img_d.parent.resolve()
            tr_rel = str(img_d.relative_to(ds_root))
            va_rel = tr_rel
        out = ds_root / "_colab_vehicle_dataset.yaml"
        body = {
            "path": str(ds_root),
            "train": tr_rel,
            "val": va_rel,
            "nc": 1,
            "names": {0: "vehicle"},
        }
        out.write_text(yaml.safe_dump(body, sort_keys=False), encoding="utf-8")
        return out
    for img_d, lbl_d in fixed_pairs:
        if _dir_has_images(img_d) and _dir_has_yolo_labels(lbl_d):
            out = root / "_colab_vehicle_dataset.yaml"
            body = {
                "path": str(root.resolve()),
                "train": str(img_d.relative_to(root)),
                "val": str(img_d.relative_to(root)),
                "nc": 1,
                "names": {0: "vehicle"},
            }
            out.write_text(yaml.safe_dump(body, sort_keys=False), encoding="utf-8")
            return out
    # Aide diagnostic
    sample = sorted({str(p.relative_to(root)) for p in root.rglob("*") if p.is_file()})[:40]
    n_txt = sum(1 for _ in root.rglob("*.txt"))
    n_xml = sum(1 for _ in root.rglob("*.xml"))
    hint = ""
    if n_xml and n_txt < 5:
        hint = " Beaucoup de fichiers XML détectés (Pascal VOC) : ce notebook attend des labels YOLO (.txt). Convertissez ou choisissez un jeu déjà au format YOLO."
    raise FileNotFoundError(
        "Aucun yaml YOLO utilisable ni structure train/images + train/labels avec labels .txt trouvée."
        + hint
        + f" Exemples de chemins sous {root} : {sample}"
    )


def _absolutize_dataset_yaml(yaml_path: Path) -> Path:
    """Réécrit un yaml avec path absolu (évite les chemins relatifs cassés sous Colab)."""
    raw = yaml_path.read_text(encoding="utf-8")
    data = yaml.safe_load(raw) or {}
    base = yaml_path.parent.resolve()
    if "path" in data and data["path"]:
        p = Path(str(data["path"]))
        if not p.is_absolute():
            data["path"] = str((base / p).resolve())
    else:
        data["path"] = str(base)
    out = yaml_path.parent / "_colab_resolved_dataset.yaml"
    out.write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return out


def download_kaggle_dataset(slug: str, dest: Path) -> Path:
    dest = Path(dest)
    shutil.rmtree(dest, ignore_errors=True)
    dest.mkdir(parents=True)
    kd = Path.home() / ".kaggle"
    kd.mkdir(parents=True, exist_ok=True)
    kaggle_json = kd / "kaggle.json"
    if not kaggle_json.is_file():
        u = os.environ.get("KAGGLE_USERNAME")
        k = os.environ.get("KAGGLE_KEY")
        if u and k:
            kaggle_json.write_text(json.dumps({"username": u, "key": k}), encoding="utf-8")
            os.chmod(kaggle_json, 0o600)
        else:
            try:
                from google.colab import userdata

                u = userdata.get("KAGGLE_USERNAME")
                k = userdata.get("KAGGLE_KEY")
                kaggle_json.write_text(json.dumps({"username": u, "key": k}), encoding="utf-8")
                os.chmod(kaggle_json, 0o600)
            except Exception as e:
                raise RuntimeError(
                    "Kaggle : remplissez KAGGLE_*_VISIBLE (cellule P), ou secrets Colab, ou ~/.kaggle/kaggle.json."
                ) from e
    zip_path = dest / "archive.zip"
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", slug, "-p", str(dest), "--force"],
        check=True,
    )
    zips = list(dest.glob("*.zip"))
    if not zips:
        return dest
    for z in zips:
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(dest)
    # Archives imbriquées (certains jeux Kaggle : zip dans zip)
    for _ in range(4):
        inner = [p for p in dest.rglob("*.zip") if p.is_file()]
        if not inner:
            break
        for zp in inner[:20]:
            try:
                with zipfile.ZipFile(zp, "r") as zf:
                    zf.extractall(zp.parent)
            except zipfile.BadZipFile:
                continue
    return dest


def download_hf_dataset(repo_id: str, dest: Path, token: Optional[str]) -> Path:
    from huggingface_hub import snapshot_download

    shutil.rmtree(dest, ignore_errors=True)
    snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        local_dir=str(dest),
        token=token,
    )
    return dest


root = Path(AERIAL_DATA_ROOT)
if AERIAL_VEHICLE_MODE.strip().lower() == "kaggle":
    download_kaggle_dataset(KAGGLE_DATASET_SLUG, root)
elif AERIAL_VEHICLE_MODE.strip().lower() == "huggingface":
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not tok:
        try:
            from google.colab import userdata

            tok = userdata.get("HF_TOKEN")
        except Exception:
            tok = None
    download_hf_dataset(HF_VEHICLE_DATASET_ID, root, tok)
else:
    raise ValueError("AERIAL_VEHICLE_MODE doit être kaggle ou huggingface")

root = _unwrap_single_dataset_folder(root)
print("Répertoire dataset (après unwrap Kaggle/HF) :", root)

yaml_p = _find_yaml_under(root)
if yaml_p is None:
    yaml_p = _ensure_minimal_yolo_yaml(root)
yaml_final = _absolutize_dataset_yaml(yaml_p)
print("dataset yaml:", yaml_final)

out_dir = Path(OUTPUT_VEHICLE_DET)
out_dir.parent.mkdir(parents=True, exist_ok=True)
cmd = [
    "yolo",
    "detect",
    "train",
    f"data={yaml_final}",
    f"model={VEHICLE_MODEL}",
    f"epochs={VEHICLE_EPOCHS}",
    f"imgsz={VEHICLE_IMGSZ}",
    f"project={out_dir.parent}",
    f"name={out_dir.name}",
]
if VEHICLE_SAVE_PERIOD and int(VEHICLE_SAVE_PERIOD) > 0:
    cmd.append(f"save_period={int(VEHICLE_SAVE_PERIOD)}")
print(" ".join(cmd))
subprocess.run(cmd, check=True)
print("Terminé — poids typiques :", out_dir / "weights" / "best.pt")